In [2]:
# import libraries

In [3]:
import urllib.request, json 
from lxml import etree
import cchardet
import requests
from bs4 import BeautifulSoup,SoupStrainer
import pkg_resources
import prettytable
import os
import re
import multiprocessing
import contextlib
from dateutil import parser

## Initial Data

In [4]:
# obtain top 8000 packages by download count during August 2023

In [4]:
with contextlib.closing(urllib.request.urlopen("https://hugovk.github.io/top-pypi-packages/top-pypi-packages-30-days.min.json")) as url:
    packages = json.load(url)
if 'top_30_packages.json' not in os.listdir('data/raw'):
    with open('data/raw/top_30_packages.json', "w") as file:
        json.dump(packages, file)
else:
    with open('data/raw/top_30_packages.json', "r") as file:
        packages = json.load(file)

In [5]:
# turn top packages into dataframe
top_packages = pd.DataFrame(packages['rows'])

<IPython.core.display.Javascript object>

In [6]:
# function to get license from a pypi project page
def getLicense(lib):
    page = requests.get(f'https://pypi.org/project/{lib}/')
    # only include html paragraph elements
    strainer = SoupStrainer('p')
    soup = BeautifulSoup(page.content, "lxml", parse_only = strainer)
    # find the element after "License:" tag
    license_text = soup.find('strong',text='License:')
    try:
        return license_text.next_sibling.strip()
    except:
        return 'No License Found'

In [9]:
%%time
# Parallelize license request process
pool = multiprocessing.Pool(12)
licenses = pool.map(getLicense, top_packages['project'])
top_packages['license'] = licenses
pool.close()

CPU times: user 153 ms, sys: 116 ms, total: 269 ms
Wall time: 2min 2s


In [10]:
# function to get github repository from a pypi project page
def getRepo(lib):
    # see if any of the url's corresponding to the project are github links
    with requests.get(f'https://pypi.org/pypi/{lib}/json') as url:
        packages = url.json()
    try:
        repos = ["/".join(ele.split("/")[:5]) for ele in packages['info']['project_urls'].values() if 'github' in ele and 'sponsors' not in ele]
        if len(repos) == 0:
            return lib
        return repos[0]
    except:
        return lib

In [11]:
%%time
pool = multiprocessing.Pool(12)
repos = pool.map(getRepo, top_packages['project'])
top_packages['repository'] = repos
pool.close()

CPU times: user 112 ms, sys: 131 ms, total: 243 ms
Wall time: 1min 26s


## Output Data - Monthly Downloads

In [13]:
# obtain list of the 8000 top repositories
top_projs = top_packages['project'].tolist()

In [14]:
# filter download data to only include these 8000 top repositories
download_info_raw = pd.read_csv('data/queries/package_downloads.csv',index_col = 0).reset_index()
download_info = download_info_raw[download_info_raw['project'].isin(top_projs)]

<IPython.core.display.Javascript object>

In [15]:
repo_dict = dict(zip(top_packages['project'], top_packages['repository']))
download_info['repository'] = download_info['project'].apply(lambda x: repo_dict[x])

/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [16]:
download_info['repository'] = download_info['repository'].apply(lambda x: x.replace("'", "").replace('.git', ''))

/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


In [17]:
download_info.sort_values(['project', 'month']).to_csv('data/clean/output_data.csv')

## Input - Labor (Monthly Contributions)

In [20]:
github_repositories = [ele for ele in download_info['repository'].drop_duplicates().apply(lambda x: "/".join(x.split("/")[-2:])).tolist() if '/'  in ele]

In [110]:
pd.Series(github_repositories).to_csv('data/queries/github_repos.csv', index = False, header = False)

<IPython.core.display.Javascript object>

In [21]:
def repoContributors(repo):
    query = f"SELECT COUNT(DISTINCT actor_login) as contributions, SUM(commits) as commit_count, SUM(additions) as additions, SUM(deletions) as deletions, DATE_TRUNC('month', created_at) as month, repo_name FROM github_events WHERE event_type == 'PullRequestEvent' AND  (repo_name = '{repo}') GROUP BY month, repo_name ORDER BY repo_name, month"
    v = os.popen(f"/Users/chrisliao/clickhouse client --secure --host play.clickhouse.com --user explorer -q \"{query}\"").read().split("\n")
    try:
        return pd.DataFrame([x.split('\t') for x in v], columns = ['contributors', 'commit count', 'additions', 'deletions', 'month', 'repo'])
    except:
        return

In [22]:
%%time
pool = multiprocessing.Pool(12)
contributor_counts1 = pd.concat(pool.map(repoContributors, github_repositories))
pool.close()

<IPython.core.display.Javascript object>

Received exception from server (version 23.8.1):
Code: 201. DB::Exception: Received from play.clickhouse.com:9440. DB::Exception: Quota for user `explorer` for 3600s has been exceeded: queries = 3001/3000. Interval will end at 2023-10-19 03:09:10. Name of quota template: `explorer`. (QUOTA_EXCEEDED)
(query: SELECT COUNT(DISTINCT actor_login) as contributions, SUM(commits) as commit_count, SUM(additions) as additions, SUM(deletions) as deletions, DATE_TRUNC('month', created_at) as month, repo_name FROM github_events WHERE event_type == 'PullRequestEvent' AND  (repo_name = 'rr-/screeninfo') GROUP BY month, repo_name ORDER BY repo_name, month)
Received exception from server (version 23.8.1):
Code: 201. DB::Exception: Received from play.clickhouse.com:9440. DB::Exception: Quota for user `explorer` for 3600s has been exceeded: queries = 3002/3000. Interval will end at 2023-10-19 03:09:10. Name of quota template: `explorer`. (QUOTA_EXCEEDED)
(query: SELECT COUNT(DISTINCT actor_login) as cont

CPU times: user 3.35 s, sys: 847 ms, total: 4.19 s
Wall time: 11min 57s


In [ ]:
uniq_repos = contributor_counts1['repo'].unique().tolist()
uncalled_repos = [ele for ele in github_repositories if ele not in uniq_repos]

In [40]:
%%time
pool = multiprocessing.Pool(12)
contributor_counts2 = pd.concat(pool.map(repoContributors, uncalled_repos))
pool.close()

<IPython.core.display.Javascript object>

CPU times: user 385 ms, sys: 296 ms, total: 681 ms
Wall time: 1min 23s


In [42]:
contributor_counts = pd.concat([contributor_counts1, contributor_counts2])

<IPython.core.display.Javascript object>

In [43]:
contributor_counts = contributor_counts[~contributor_counts['month'].isna()]

In [45]:
contributor_counts.to_csv('data/clean/contributor_counts.csv')

## Input - Capital (Issue Resolution)

In [194]:
issues_summary = pd.concat([pd.read_csv('data/queries/github_issues_2015_22.csv'),
                            pd.read_csv('data/queries/github_issues_2023.csv')]).reset_index(drop = True)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [195]:
issues_summary['state'] = issues_summary['state'].apply(lambda x: x.replace("\"", ""))

In [201]:
issues_summary['earliest_date'].apply(lambda x: len(x)).value_counts()

23    2484951
Name: earliest_date, dtype: int64

In [203]:
issues_summary['latest_date'] = pd.to_datetime(issues_summary['latest_date'])
issues_summary['earliest_date'] = pd.to_datetime(issues_summary['earliest_date'])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [206]:
for close_type in ['early', 'late']:
    if close_type == 'early':
        issues_summary['date'] = issues_summary.apply(lambda x: x['latest_date'] if x['state'] == 'closed' else x['earliest_date'], axis = 1)
    else:
        issues_summary['date'] = issues_summary.apply(lambda x: x['earliest_date'] if x['state'] == 'closed' else x['earliest_date'], axis = 1)
    
    issues_summary = issues_summary.sort_values('date').drop_duplicates(['name','number','state'], keep = 'last')

    df_issues_summary = issues_summary.drop(['latest_date', 'earliest_date'], axis = 1).pivot(index=['name', 'number'], 
                                                                                   columns='state', values='date').reset_index()

    df_issues_summary['resolution time'] = df_issues_summary['closed'] - df_issues_summary['open']
    df_issues_summary['date'] = df_issues_summary['open'].dt.to_period('m')
    df_issues_summary.to_csv(f'data/clean/issue_resolution_{close_type}.csv')

/opt/anaconda3/lib/python3.7/site-packages/pandas/core/arrays/datetimes.py:1146: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  UserWarning,
/opt/anaconda3/lib/python3.7/site-packages/pandas/core/arrays/datetimes.py:1146: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  UserWarning,
